# BoW·DTM·TF-IDF 문서 벡터화

컴퓨터 모델은 문장을 문자열 그대로 비교할 수 없으므로 문서마다 같은 어휘 축을 사용하는 수치 벡터가 필요하다.
**BoW(Bag of Words)** 는 각 단어의 등장 횟수를 기록하고,
**TF-IDF(Term Frequency-Inverse Document Frequency)** 는 여러 문서에 흔한 단어의 영향력을 낮춰 문서를 구별하는 단어를 더 강조한다.
두 방법 모두 결과를 단어 열과 직접 연결해 읽을 수 있어 검색·문서 분류의 빠르고 해석 가능한 기준선에 자주 사용한다.

이 노트북에서는 다섯 문장을 같은 어휘 축의 DTM으로 바꾸고, 행과 열을 전치한 TDM을 구분한 뒤 BoW와 TF-IDF 유사도를 비교한다.
마지막 결과에서는 같은 문장 쌍도 단어 가중치가 달라지면 코사인 유사도가 어떻게 변하는지 확인하며, 빈도 표현이 단어 순서와 문맥 의미를 잃는다는 다음 임베딩 기법의 도입 이유까지 연결한다.


## 01. 예제 문서

**문서(Document)** 는 하나의 벡터로 변환할 텍스트 단위이고, **코퍼스(Corpus)** 는 같은 분석 목적에 따라 모은 문서의 집합이다.
아래 `sentences`에서는 리스트 원소 하나가 문서 하나이며, 출력되는 `sent1`부터 `sent5`가 이후 행렬의 행 순서와 그대로 대응한다.

같은 `love`, `my`, `dog`가 여러 문서에 반복되고 `cat`, `amazing`처럼 일부 문서에만 있는 단어도 있다. 이 의도적인 차이 덕분에 다음 코드에서 단순 등장 횟수와 코퍼스 전체를 고려한 중요도 가중치가 어떻게 달라지는지 비교할 수 있다.

In [20]:
# sentences 전체 == 코퍼스(Corpus)
# sentences 안에 들어있는 각 요소 == Document

sentences = [
    "I love my dog.",
    # "I dog love my.",
    "I love my cat.",
    "I love my dog and love my cat.",
    "You love my dog!",
    "Do you think my dog is amazing?",
]

for document_id, sentence in enumerate(sentences, start=1):
    print(f"sent{document_id}: {sentence}")


sent1: I love my dog.
sent2: I love my cat.
sent3: I love my dog and love my cat.
sent4: You love my dog!
sent5: Do you think my dog is amazing?


## 02. BoW와 문서-단어 행렬

**BoW(Bag of Words)** 는 단어 순서를 무시하고 문서마다 어휘가 등장한 횟수를 세는 표현이다.
각 문서를 행, 코퍼스에서 얻은 고유 단어를 열로 놓으면 **DTM(Document-Term Matrix)** 이 되고, 한 칸의 값은 해당 행의 문서에서 해당 열의 단어가 나타난 횟수가 된다.
어휘가 커질수록 문서마다 사용하지 않은 단어 열이 많아지므로 DTM은 대부분의 값이 0인 **희소 행렬(Sparse Matrix)** 로 저장하는 것이 효율적이다.

`CountVectorizer.fit_transform(sentences)`는 문서를 토큰화해 **어휘 사전(Vocabulary)** 의 단어-열 번호 대응을 학습하고, 같은 규칙으로 다섯 문서를 `(문서 수, 어휘 수)` 행렬로 변환한다.
다음 코드를 실행해 shape이 `(5, 10)`이면 문서 5개가 단어 열 10개로 표현됐다는 뜻이다.
표에서 `sent3`의 `love`와 `my`가 각각 2이고 문장에 없는 단어 열은 0인지 확인하면 행·열·값의 의미를 실제 원문과 연결할 수 있다.

In [21]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

count_vectorizer = CountVectorizer()
bow_sparse = count_vectorizer.fit_transform(sentences)
# print(bow_sparse)

# 열 번호에 대응하는 단어를 순서대로 가져오기
feature_names = count_vectorizer.get_feature_names_out()
print(feature_names)

bow_df = pd.DataFrame(
    bow_sparse.toarray(),
    index=[f'sent{i}' for i in range(1, len(sentences) + 1)],
    columns=feature_names,
)

print('BoW shape:', bow_df.shape)
display(bow_df)



['amazing' 'and' 'cat' 'do' 'dog' 'is' 'love' 'my' 'think' 'you']
BoW shape: (5, 10)


,amazing,and,cat,do,dog,is,love,my,think,you
sent1,0,0,0,0,1,0,1,1,0,0
sent2,0,0,1,0,0,0,1,1,0,0
sent3,0,1,1,0,1,0,2,2,0,0
sent4,0,0,0,0,1,0,1,1,0,1
sent5,1,0,0,1,1,1,0,1,1,1


### DTM을 TDM으로 전치하기

앞의 표는 `문서 × 단어`인 DTM이다. 이를 전치한 **TDM(Term-Document Matrix)** 은 행이 단어이고 열이 문서인 `단어 × 문서` 행렬이다.
두 이름은 새로운 가중치 계산법을 뜻하지 않고 같은 빈도표를 어느 축에서 읽는지 구분한다.

`DataFrame.T`는 2차원 표의 행과 열을 맞바꾸므로 shape은 `(5, 10)`에서 `(10, 5)`로 바뀐다. 그러나 `sent3`과 `love`가 만나는 값은 두 표에서 모두 2여야 하며, 실행 후 이 조건을 만족하면 전치가 값이 아니라 축의 관점만 바꿨다고 판단할 수 있다.

In [22]:
tdm_df = bow_df.T

print("DTM shape:", bow_df.shape)
print("TDM shape:", tdm_df.shape)
print("sent3의 love 빈도:", bow_df.loc["sent3", "love"])
print("love의 sent3 빈도:", tdm_df.loc["love", "sent3"])
display(tdm_df)

DTM shape: (5, 10)
TDM shape: (10, 5)
sent3의 love 빈도: 2
love의 sent3 빈도: 2


,sent1,sent2,sent3,sent4,sent5
amazing,0,0,0,0,1
and,0,0,1,0,0
cat,0,1,1,0,0
do,0,0,0,0,1
dog,1,0,1,1,1
is,0,0,0,0,1
love,1,1,2,1,0
my,1,1,2,1,1
think,0,0,0,0,1
you,0,0,0,1,1


### BoW 문서 벡터의 코사인 유사도

코사인 유사도는 두 벡터의 절대 길이보다 방향을 비교한다. BoW에서는 공유하는 단어와 그 빈도 패턴이 비슷할수록 값이 커진다.

$$
\operatorname{cosine}(x,y)=\frac{x\cdot y}{\lVert x\rVert_2\lVert y\rVert_2}
$$

값은 보통 `-1`에서 `1` 사이이며 현재처럼 음수가 없는 빈도 벡터에서는 `0`에서 `1` 사이가 된다. 대각선은 같은 문서를 자신과 비교하므로 1이다. 실행 후 `(5, 5)` 행렬에서 `sent1-sent3`이 `sent1-sent5`보다 큰지 확인한다. 앞의 두 문서가 `dog`, `love`, `my`를 함께 사용하기 때문에 예상되는 비교 방향이며, 높은 값만으로 문장 의미까지 같다고 보장할 수 없다.

In [23]:
from sklearn.metrics.pairwise import cosine_similarity

# 코퍼스 == 문장 5개 묶음
# 다큐먼트 == 각 문장 1개
# 문장별과 단어별로 코사인 유사도를 측정
bow_similarity = cosine_similarity(bow_sparse)
bow_similarity_df = pd.DataFrame(
    bow_similarity,
    index=bow_df.index,  # sent1 ~ sent5
    columns=bow_df.index, # sent1 ~ sent5
)

display(bow_similarity_df.round(3))


# 코사인 유사도 수치가 높다는 뜻은 같은 문장이라는 뜻이 아님. 그저 문장의 단어 사용 패턴이 유사하다는 뜻으로 해석. 같은 단어가 많이 나올수록 유사도가 높은데 단어 순서는 상관없이 같은 단어가 나오기만 하면 유사도가 높음.


,sent1,sent2,sent3,sent4,sent5
sent1,1.000,0.667,0.870,0.866,0.436
sent2,0.667,1.000,0.870,0.577,0.218
sent3,0.870,0.870,1.000,0.754,0.342
sent4,0.866,0.577,0.754,1.000,0.567
sent5,0.436,0.218,0.342,0.567,1.000


## 03. TF-IDF

BoW는 자주 등장한 단어에 큰 값을 주지만 모든 문서에 반복되는 단어는 문서를 구별하는 데 도움이 적을 수 있다.
**TF(Term Frequency)** 는 단어 $t$가 문서 $d$ 안에 나타난 빈도이고, **DF(Document Frequency)** 는 그 단어가 등장한 문서의 수이다. **IDF(Inverse Document Frequency)** 는 DF가 큰 흔한 단어의 가중치를 낮추며,
TF와 IDF를 곱한 **TF-IDF** 는 한 문서에서는 자주 등장하지만 전체 코퍼스에서는 드문 단어를 강조한다.
강조된 단어는 문서를 구분하는 용도로 사용된다.

문서가 길면 모든 단어의 원시 가중치가 함께 커질 수 있으므로 `TfidfVectorizer`는 기본적으로 각 문서 벡터의 유클리드 길이를 1로 맞추는 **L2 정규화**까지 수행한다. 따라서 BoW와 TF-IDF는 같은 어휘 축과 같은 `(5, 10)` shape을 사용해도, 전자는 횟수이고 후자는 코퍼스의 문서 빈도와 문서 길이를 함께 반영한 실수 가중치라는 점이 다르다.

scikit-learn의 기본 smoothed IDF는 다음 식을 사용한다.

$$
\operatorname{idf}(t)=\log\left(\frac{1+N}{1+\operatorname{df}(t)}\right)+1
$$

$N$은 전체 문서 수이다. 실행 후 `sent2` 표에서 가중치가 `cat > love > my` 순서인지 확인한다. 세 단어의 등장 횟수는 같지만 각각 등장한 문서 수가 2, 4, 5이므로 더 드문 `cat`이 큰 값을 가지면 IDF가 필요한 이유를 읽을 수 있다.

In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TfidfVectorizer()를 이용해 어휘(단어)와 IDF를 학습하고 벡터화
tfidf_vectorizer = TfidfVectorizer()
tfidf_sparse = tfidf_vectorizer.fit_transform(sentences)
tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()
tfidf_df = pd.DataFrame(
    tfidf_sparse.toarray(),
    index=bow_df.index, # sent1 ~ sent5
    columns=tfdif_feature_names,
)

tfidf_df

# TF-IDF에 표기되는 실수는 빈도수 X, 가중치 O
# 전체 Document에서 자주 등장하는 my, love의 가중치는 낮게,   일부에서만 등장하는 cat, amazing은 가중치가 높게 표현된다.

,amazing,and,cat,do,dog,is,love,my,think,you
sent1,0.000000,0.000000,0.000000,0.000000,0.606856,0.000000,0.606856,0.513275,0.000000,0.000000
sent2,0.000000,0.000000,0.737922,0.000000,0.000000,0.000000,0.515290,0.435829,0.000000,0.000000
sent3,0.000000,0.491109,0.396224,0.000000,0.276682,0.000000,0.553364,0.468032,0.000000,0.000000
sent4,0.000000,0.000000,0.000000,0.000000,0.458054,0.000000,0.458054,0.387419,0.000000,0.655957
sent5,0.438724,0.000000,0.000000,0.438724,0.247170,0.438724,0.000000,0.209054,0.438724,0.353960


### `sent2`의 TF-IDF 직접 계산

`sent2`에는 `cat`, `love`, `my`가 한 번씩 등장한다. scikit-learn의 기본 TF는 문서 길이로 나눈 비율이 아니라 원래 단어 빈도이므로, 이 문서에서 정규화 전 TF-IDF는 각 단어의 IDF와 같다. `np.log()`로 smoothed IDF를 계산하고 `np.linalg.norm()`으로 구한 L2 길이로 나누면 `TfidfVectorizer`의 기본 출력 과정을 직접 재현할 수 있다.

직접 계산한 세 값과 라이브러리 표의 같은 위치를 `np.allclose()`로 비교한다. 출력의 DF가 `cat=2`, `love=4`, `my=5`이고 직접 계산과 라이브러리 값이 각각 `0.737922`, `0.515290`, `0.435829`로 일치하며 마지막 결과가 `True`라면, TF·IDF·L2 정규화의 순서를 올바르게 연결한 것이다.

In [25]:
import numpy as np

target_words = ["cat", "love", "my"]
document_count = len(sentences)

document_frequencies = {
    word: int((bow_df[word] > 0).sum())
    for word in target_words
}

idf_values = np.array([
    np.log((1 + document_count) / (1 + document_frequencies[word])) + 1
    for word in target_words
])

raw_tfidf = idf_values
manual_tfidf = raw_tfidf / np.linalg.norm(raw_tfidf)


library_tfidf = tfidf_df.loc["sent2", target_words].to_numpy()

comparison_df = pd.DataFrame(
    {
        "DF": [document_frequencies[word] for word in target_words],
        "IDF": idf_values,
        "직접 계산": manual_tfidf,
        "라이브러리": library_tfidf,
    },
    index=target_words,
)

display(comparison_df.round(6))
print("같은 값:", np.allclose(manual_tfidf, library_tfidf))

,DF,IDF,직접 계산,라이브러리
cat,2,1.693147,0.737922,0.737922
love,4,1.182322,0.515290,0.515290
my,5,1.000000,0.435829,0.435829


같은 값: True


## 04. BoW와 TF-IDF 유사도 비교

두 표현은 같은 문장과 같은 어휘를 사용하지만 단어 가중치가 달라 문서 사이의 코사인 유사도도 달라진다. TF-IDF에서는 코퍼스 전체에 흔한 `my`, `love`의 영향이 줄고 비교적 드문 단어가 문서 방향에 더 크게 반영된다.

BoW는 빈도 자체가 중요한 짧은 기준선과 해석 가능한 모델에 적합하다. TF-IDF는 검색, 키워드 추출, 문서 분류처럼 문서를 구별하는 단어를 강조할 때 먼저 고려한다. 아래 비교에서 세 문장 쌍 모두 TF-IDF 유사도가 BoW보다 낮아지면 흔한 `my`, `love`가 만들던 공통 방향의 영향이 줄었다고 해석할 수 있다. 유사도 수치가 더 높거나 낮다는 사실만으로 표현의 우열을 정하지 않고 실제 분류·검색 성능으로 선택한다.

두 방식 모두 단어가 몇 번 등장했는지는 보존하지만 단어 순서와 문맥에 따른 의미 변화는 직접 표현하지 못한다.

In [26]:
tfidf_similarity = cosine_similarity(tfidf_sparse)
tfidf_similarity_df = pd.DataFrame(
    tfidf_similarity,
    index=tfidf_df.index,
    columns=tfidf_df.index,
)

comparison = pd.DataFrame(
    {
        "BoW": [
            bow_similarity_df.loc["sent1", "sent3"],
            bow_similarity_df.loc["sent1", "sent5"],
            bow_similarity_df.loc["sent2", "sent5"],
        ],
        "TF-IDF": [
            tfidf_similarity_df.loc["sent1", "sent3"],
            tfidf_similarity_df.loc["sent1", "sent5"],
            tfidf_similarity_df.loc["sent2", "sent5"],
        ],
    },
    index=["sent1-sent3", "sent1-sent5", "sent2-sent5"],
)

display(comparison.round(3))

# TF-IDF는 자주 등장하는 단어의 가중치를 줄였기 때문에  단어의 유사도가 낮아져서 코사인 유사도가 BoW보다 낮게 측정된다.

# BoW와 TF-IDF 중 어떤 표현이 좋다기보단 목표 작업에 맞춰 적절한 방식을 선택해야만 한다.

# (예전 네이버실시간검색 같은 건 BoW가 맞는 거고, 주요 키워드 빈도를 보고 싶을 때는 TF-IDF가 맞다.)


,BoW,TF-IDF
sent1-sent3,0.870,0.744
sent1-sent5,0.436,0.257
sent2-sent5,0.218,0.091
